# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [2]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [3]:
cleaned_path = "zillow_cleaned.csv"

if not os.path.exists(cleaned_path):
    raise FileNotFoundError(
        f"{cleaned_path} was not found. In Milestone_01.ipynb, rerun your cleaning steps through Part 3 and save the result with:\n"
        'df_cleaned.to_csv("zillow_cleaned.csv", index=False)'
    )

df = pd.read_csv(cleaned_path)
print("Loaded cleaned dataset:", df.shape)
display(df.head())

target = "taxvaluedollarcnt"
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Encode any remaining categorical columns after the split to avoid leakage.
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
if categorical_cols:
    print("Encoding categorical columns:", categorical_cols)
    X_train = pd.get_dummies(X_train, columns=categorical_cols, dummy_na=True)
    X_test = pd.get_dummies(X_test, columns=categorical_cols, dummy_na=True)
    X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

# Ensure all remaining values are numeric before scaling.
X_train = X_train.apply(pd.to_numeric, errors="coerce")
X_test = X_test.apply(pd.to_numeric, errors="coerce")

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
display(X_train_scaled.head())


Loaded cleaned dataset: (77572, 20)


,bathroomcnt,bedroomcnt,buildingqualitytypeid,calculatedbathnbr,calculatedfinishedsquarefeet,finishedsquarefeet12,fullbathcnt,heatingorsystemtypeid,latitude,longitude,lotsizesquarefeet,propertycountylandusecode,propertylandusetypeid,propertyzoningdesc,regionidcity,regionidzip,roomcnt,unitcnt,yearbuilt,taxvaluedollarcnt
0,3.5,4.0,6.0,3.5,3100.0,3100.0,3.0,2.0,33634931.0,-117869207.0,4506.0,60,261.0,580,53571.0,96978.0,0.0,1.0,1998.0,1023282.0
1,1.0,2.0,6.0,1.0,1465.0,1465.0,1.0,2.0,34449266.0,-119281531.0,12647.0,51,261.0,580,13091.0,97099.0,5.0,1.0,1967.0,464000.0
2,2.0,3.0,6.0,2.0,1243.0,1243.0,2.0,2.0,33886168.0,-117823170.0,8432.0,60,261.0,580,21412.0,97078.0,6.0,1.0,1962.0,564778.0
3,3.0,4.0,8.0,3.0,2376.0,2376.0,3.0,2.0,34245180.0,-118240722.0,13038.0,1,261.0,797,396551.0,96330.0,0.0,1.0,1970.0,145143.0
4,3.0,3.0,8.0,3.0,1312.0,1312.0,3.0,2.0,34185120.0,-118414640.0,278581.0,8,266.0,587,12447.0,96451.0,0.0,1.0,1964.0,119407.0


X_train shape: (62057, 19)
X_test shape: (15515, 19)
y_train shape: (62057,)
y_test shape: (15515,)


,bathroomcnt,bedroomcnt,buildingqualitytypeid,calculatedbathnbr,calculatedfinishedsquarefeet,finishedsquarefeet12,fullbathcnt,heatingorsystemtypeid,latitude,longitude,lotsizesquarefeet,propertycountylandusecode,propertylandusetypeid,propertyzoningdesc,regionidcity,regionidzip,roomcnt,unitcnt,yearbuilt
73889,0.199659,-0.045796,-0.244508,0.188019,0.018639,0.058510,-0.259516,-0.4067,-1.211538,1.274971,-0.168696,0.920277,0.800108,-0.432868,0.405283,0.094306,-0.520575,-0.071353,1.741508
278,0.699872,-0.045796,-0.244508,0.698717,1.351702,1.452769,0.775563,-0.4067,0.881284,-1.919013,0.145226,1.029950,-0.156079,-0.432868,0.375274,0.130225,1.967677,-0.071353,0.859322
30994,0.699872,1.704711,1.178091,0.698717,0.152363,0.198373,0.775563,-0.4067,-0.775222,-0.318483,-0.187500,-0.871055,-0.156079,0.754642,0.413301,-0.095627,-0.520575,-0.071353,-0.148892
58858,0.199659,0.829458,-0.244508,0.188019,-0.028374,0.009340,-0.259516,-0.4067,1.076271,-2.653498,-0.168696,1.249297,0.800108,-0.432868,0.020186,0.128416,1.967677,-0.071353,1.909544
55015,-0.300553,-0.045796,1.889390,-0.322678,0.532641,0.596108,-0.259516,-0.4067,-0.894877,-0.438489,-0.060650,-0.871055,-0.156079,0.425635,-0.005995,-0.120435,-0.520575,-0.071353,-0.358936


### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [4]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

required_vars = ["X_train_scaled", "X_test_scaled", "y_train", "y_test"]
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise NameError(f"Run the Prelude cell first. Missing variables: {missing_vars}")

# Pick three baseline models from the approved list.
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Random Forest": RandomForestRegressor(random_state=random_state),
}

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
results = []
fitted_models = {}

for model_name, model in models.items():
    cv_scores = -cross_val_score(
        model,
        X_train_scaled,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1,
    )

    model.fit(X_train_scaled, y_train)
    fitted_models[model_name] = model

    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)

    results.append({
        "Model": model_name,
        "CV MAE Mean": cv_scores.mean(),
        "CV MAE Std": cv_scores.std(),
        "Train MAE": mean_absolute_error(y_train, train_pred),
        "Test MAE": mean_absolute_error(y_test, test_pred),
        "Train RMSE": mean_squared_error(y_train, train_pred) ** 0.5,
        "Test RMSE": mean_squared_error(y_test, test_pred) ** 0.5,
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred),
    })

part1_results = pd.DataFrame(results).sort_values(by="CV MAE Mean")
display(part1_results.round({
    "CV MAE Mean": 2,
    "CV MAE Std": 2,
    "Train MAE": 2,
    "Test MAE": 2,
    "Train RMSE": 2,
    "Test RMSE": 2,
    "Train R2": 4,
    "Test R2": 4,
}))

best_model_name = part1_results.iloc[0]["Model"]
print(f"Best baseline by CV MAE: {best_model_name}")



/Users/bunphengchhay/miniforge3/envs/bu_env/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,Model,CV MAE Mean,CV MAE Std,Train MAE,Test MAE,Train RMSE,Test RMSE,Train R2,Test R2
2,Random Forest,191427.73,3017.52,71741.83,189378.79,151940.21,482062.24,0.9427,0.5569
1,Ridge Regression,246553.38,3388.92,246292.61,242250.12,486491.65,564146.50,0.4122,0.3931
0,Linear Regression,246555.22,3388.94,246294.03,242251.44,486491.65,564145.89,0.4122,0.3931


Best baseline by CV MAE: Random Forest


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

For Part 1, the Random Forest model performed the best overall. It had the lowest cross-validation MAE mean at about 191,428, which was much better than both Ridge Regression and Linear Regression, which were both around 246,555. Random Forest also had the best test performance, with a test MAE of about 189,379 and a test R² of 0.5569, so it explained more of the variation in housing values than the two linear models.

The most stable model was Random Forest, since it had the lowest CV standard deviation (3017.52) compared with Ridge and Linear Regression (both about 3389). This suggests its performance was a little more consistent across the repeated cross-validation splits. Ridge Regression and Linear Regression performed almost identically in every metric, which makes sense because the default Ridge penalty was not strong enough to create much difference from ordinary linear regression on this dataset.

There are also some signs of overfitting, especially with Random Forest. Its training MAE (71,741.83) is much lower than its test MAE (189,378.79), and its training R² (0.9427) is much higher than its test R² (0.5569). This gap suggests the model fits the training data very well but does not generalize as strongly to unseen data. In contrast, Linear Regression and Ridge Regression had train and test results that were much closer together, so they appear less overfit, but their overall performance was also clearly worse. Overall, Random Forest is the strongest baseline so far, but it may benefit from tuning in later parts to reduce overfitting.

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make sure the Prelude cell has already been run.
required_vars = ["X_train", "X_test", "y_train", "y_test"]
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise NameError(f"Run the Prelude cell first. Missing variables: {missing_vars}")

# Start from the cleaned features created in the Prelude.
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

# New engineered features.
# 1. Nonlinear square-footage term from Milestone 1 polynomial-feature findings.
X_train_fe["sqft_squared"] = X_train_fe["calculatedfinishedsquarefeet"] ** 2
X_test_fe["sqft_squared"] = X_test_fe["calculatedfinishedsquarefeet"] ** 2

# 2. Age of the home instead of raw construction year.
reference_year = 2016
X_train_fe["home_age"] = reference_year - X_train_fe["yearbuilt"]
X_test_fe["home_age"] = reference_year - X_test_fe["yearbuilt"]

# 3. Interaction between bathroom count and living area.
X_train_fe["bath_sqft_interaction"] = (
    X_train_fe["bathroomcnt"] * X_train_fe["calculatedfinishedsquarefeet"]
)
X_test_fe["bath_sqft_interaction"] = (
    X_test_fe["bathroomcnt"] * X_test_fe["calculatedfinishedsquarefeet"]
)

# 4. Ratio of lot size to living area.
X_train_fe["lot_living_ratio"] = (
    X_train_fe["lotsizesquarefeet"] / X_train_fe["calculatedfinishedsquarefeet"].replace(0, np.nan)
)
X_test_fe["lot_living_ratio"] = (
    X_test_fe["lotsizesquarefeet"] / X_test_fe["calculatedfinishedsquarefeet"].replace(0, np.nan)
)

# Clean up any missing or infinite values introduced by feature engineering.
X_train_fe = X_train_fe.replace([np.inf, -np.inf], np.nan)
X_test_fe = X_test_fe.replace([np.inf, -np.inf], np.nan)

fe_imputer = SimpleImputer(strategy="median")
X_train_fe = pd.DataFrame(
    fe_imputer.fit_transform(X_train_fe),
    columns=X_train_fe.columns,
    index=X_train_fe.index,
)
X_test_fe = pd.DataFrame(
    fe_imputer.transform(X_test_fe),
    columns=X_test_fe.columns,
    index=X_test_fe.index,
)

scaler_fe = StandardScaler()
X_train_fe_scaled = pd.DataFrame(
    scaler_fe.fit_transform(X_train_fe),
    columns=X_train_fe.columns,
    index=X_train_fe.index,
)
X_test_fe_scaled = pd.DataFrame(
    scaler_fe.transform(X_test_fe),
    columns=X_test_fe.columns,
    index=X_test_fe.index,
)

print("Engineered features added:")
print(["sqft_squared", "home_age", "bath_sqft_interaction", "lot_living_ratio"])
print("Remaining NaNs in X_train_fe_scaled:", int(X_train_fe_scaled.isna().sum().sum()))
print("Remaining NaNs in X_test_fe_scaled:", int(X_test_fe_scaled.isna().sum().sum()))

models_fe = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Random Forest": RandomForestRegressor(random_state=random_state),
}

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
results_fe = []

for model_name, model in models_fe.items():
    cv_scores = -cross_val_score(
        model,
        X_train_fe_scaled,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1,
    )

    model.fit(X_train_fe_scaled, y_train)
    train_pred = model.predict(X_train_fe_scaled)
    test_pred = model.predict(X_test_fe_scaled)

    results_fe.append({
        "Model": model_name,
        "CV MAE Mean": cv_scores.mean(),
        "CV MAE Std": cv_scores.std(),
        "Train MAE": mean_absolute_error(y_train, train_pred),
        "Test MAE": mean_absolute_error(y_test, test_pred),
        "Train RMSE": mean_squared_error(y_train, train_pred) ** 0.5,
        "Test RMSE": mean_squared_error(y_test, test_pred) ** 0.5,
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred),
    })

part2_results = pd.DataFrame(results_fe).sort_values(by="CV MAE Mean")
display(part2_results.round({
    "CV MAE Mean": 2,
    "CV MAE Std": 2,
    "Train MAE": 2,
    "Test MAE": 2,
    "Train RMSE": 2,
    "Test RMSE": 2,
    "Train R2": 4,
    "Test R2": 4,
}))

best_model_part2 = part2_results.iloc[0]["Model"]
print(f"Best Part 2 model by CV MAE: {best_model_part2}")


Engineered features added:
['sqft_squared', 'home_age', 'bath_sqft_interaction', 'lot_living_ratio']
Remaining NaNs in X_train_fe_scaled: 0
Remaining NaNs in X_test_fe_scaled: 0


/Users/bunphengchhay/miniforge3/envs/bu_env/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,Model,CV MAE Mean,CV MAE Std,Train MAE,Test MAE,Train RMSE,Test RMSE,Train R2,Test R2
2,Random Forest,191822.75,3126.71,72034.92,189730.71,152864.64,481547.76,0.9420,0.5578
0,Linear Regression,233837.55,3530.53,233416.79,228015.81,472370.49,490204.24,0.4458,0.5418
1,Ridge Regression,233837.70,3530.48,233416.81,228015.44,472370.49,490213.19,0.4458,0.5418


Best Part 2 model by CV MAE: Random Forest


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




The new engineered features had a mixed impact on model performance. The biggest improvement was for the linear models. In Part 1, both Linear Regression and Ridge Regression had CV MAE values around 246,555, but after adding the new features, their CV MAE dropped to about 233,838. Their test performance also improved, with test R² increasing to about 0.5418, which is much better than before. This suggests the engineered features helped the linear models capture relationships that were not represented well by the original features alone.

The features that seemed most helpful were probably the more nonlinear and interaction-based features, especially sqft_squared and bath_sqft_interaction. These likely helped Linear Regression and Ridge Regression because those models can only learn linear relationships unless we manually add transformed features. By including squared square footage and an interaction between bathrooms and living area, the models were able to represent more complex patterns in housing prices. The home_age feature also likely helped because age is easier to interpret than raw year built and may connect more directly to property value. The lot_living_ratio feature may have added some useful information, but based on the overall results, it seems less impactful than the size-related features.

For Random Forest, the new features did not lead to clear improvement. Its CV MAE actually became slightly worse, going from about 191,428 in Part 1 to 191,823 in Part 2, although the test R² improved a little from 0.5569 to 0.5578. My hypothesis is that Random Forest already captures nonlinear relationships and interactions on its own, so manually engineered features do not help it as much as they help simpler linear models. Overall, feature engineering was most useful for the linear models, while Random Forest remained the best-performing model overall but benefited less from the added features.

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [6]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make sure Part 2 has already been run.
required_vars = ["X_train_fe", "X_test_fe", "y_train", "y_test"]
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise NameError(f"Run the Part 2 feature engineering cell first. Missing variables: {missing_vars}")

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

# Candidate subset sizes to test.
n_features_total = X_train_fe.shape[1]
candidate_ks = sorted(set([
    min(5, n_features_total),
    min(8, n_features_total),
    min(10, n_features_total),
    min(12, n_features_total),
    min(15, n_features_total),
    n_features_total,
]))

models_fs = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Random Forest": RandomForestRegressor(random_state=random_state),
}

part3_results = []
best_feature_subsets = {}

for model_name, model in models_fs.items():
    best_cv_mae = np.inf
    best_cv_std = None
    best_k = None
    best_features = None
    best_train_matrix = None
    best_test_matrix = None

    for k in candidate_ks:
        if model_name in ["Linear Regression", "Ridge Regression"]:
            selector = SelectKBest(score_func=f_regression, k=k)
            X_train_selected = selector.fit_transform(X_train_fe, y_train)
            X_test_selected = selector.transform(X_test_fe)
            selected_features = X_train_fe.columns[selector.get_support()].tolist()

            scaler_fs = StandardScaler()
            X_train_selected = scaler_fs.fit_transform(X_train_selected)
            X_test_selected = scaler_fs.transform(X_test_selected)
        else:
            importance_model = RandomForestRegressor(random_state=random_state)
            importance_model.fit(X_train_fe, y_train)
            importances = pd.Series(importance_model.feature_importances_, index=X_train_fe.columns)
            selected_features = importances.sort_values(ascending=False).head(k).index.tolist()
            X_train_selected = X_train_fe[selected_features].copy()
            X_test_selected = X_test_fe[selected_features].copy()

        cv_scores = -cross_val_score(
            model,
            X_train_selected,
            y_train,
            scoring="neg_mean_absolute_error",
            cv=cv,
            n_jobs=-1,
        )

        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()

        if cv_mean < best_cv_mae:
            best_cv_mae = cv_mean
            best_cv_std = cv_std
            best_k = k
            best_features = selected_features
            best_train_matrix = X_train_selected
            best_test_matrix = X_test_selected

    best_feature_subsets[model_name] = best_features

    model.fit(best_train_matrix, y_train)
    train_pred = model.predict(best_train_matrix)
    test_pred = model.predict(best_test_matrix)

    part3_results.append({
        "Model": model_name,
        "Best k": best_k,
        "Selected Features": ", ".join(best_features),
        "CV MAE Mean": best_cv_mae,
        "CV MAE Std": best_cv_std,
        "Train MAE": mean_absolute_error(y_train, train_pred),
        "Test MAE": mean_absolute_error(y_test, test_pred),
        "Train RMSE": mean_squared_error(y_train, train_pred) ** 0.5,
        "Test RMSE": mean_squared_error(y_test, test_pred) ** 0.5,
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred),
    })

part3_results = pd.DataFrame(part3_results).sort_values(by="CV MAE Mean")
display(part3_results.round({
    "CV MAE Mean": 2,
    "CV MAE Std": 2,
    "Train MAE": 2,
    "Test MAE": 2,
    "Train RMSE": 2,
    "Test RMSE": 2,
    "Train R2": 4,
    "Test R2": 4,
}))

print("\nBest feature subsets by model:")
for model_name, features in best_feature_subsets.items():
    print(f"\n{model_name} ({len(features)} features):")
    print(features)


/Users/bunphengchhay/miniforge3/envs/bu_env/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,Model,Best k,Selected Features,CV MAE Mean,CV MAE Std,Train MAE,Test MAE,Train RMSE,Test RMSE,Train R2,Test R2
2,Random Forest,23,"finishedsquarefeet12, regionidzip, latitude, l...",191879.33,3060.21,72015.02,189227.63,152440.25,481646.30,0.9423,0.5576
0,Linear Regression,23,"bathroomcnt, bedroomcnt, buildingqualitytypeid...",233837.55,3530.53,233416.79,228015.81,472370.49,490204.24,0.4458,0.5418
1,Ridge Regression,23,"bathroomcnt, bedroomcnt, buildingqualitytypeid...",233837.70,3530.48,233416.81,228015.44,472370.49,490213.19,0.4458,0.5418



Best feature subsets by model:

Linear Regression (23 features):
['bathroomcnt', 'bedroomcnt', 'buildingqualitytypeid', 'calculatedbathnbr', 'calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'fullbathcnt', 'heatingorsystemtypeid', 'latitude', 'longitude', 'lotsizesquarefeet', 'propertycountylandusecode', 'propertylandusetypeid', 'propertyzoningdesc', 'regionidcity', 'regionidzip', 'roomcnt', 'unitcnt', 'yearbuilt', 'sqft_squared', 'home_age', 'bath_sqft_interaction', 'lot_living_ratio']

Ridge Regression (23 features):
['bathroomcnt', 'bedroomcnt', 'buildingqualitytypeid', 'calculatedbathnbr', 'calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'fullbathcnt', 'heatingorsystemtypeid', 'latitude', 'longitude', 'lotsizesquarefeet', 'propertycountylandusecode', 'propertylandusetypeid', 'propertyzoningdesc', 'regionidcity', 'regionidzip', 'roomcnt', 'unitcnt', 'yearbuilt', 'sqft_squared', 'home_age', 'bath_sqft_interaction', 'lot_living_ratio']

Random Forest (23 features):
['fin

### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


Feature selection did not really improve performance for any of the models. The CV MAE scores were almost the same as in Part 2, which suggests that reducing the number of features did not make a major difference. Random Forest still performed best overall, while Linear Regression and Ridge Regression stayed very close to each other.

The features that were consistently retained across models were mostly size, location, and quality related features, such as calculatedfinishedsquarefeet, finishedsquarefeet12, latitude, longitude, lotsizesquarefeet, yearbuilt, bathroomcnt, bedroomcnt, and regionidzip. These make sense because they are closely related to property value.

Yes, the engineered features were also selected as important. Features like sqft_squared, home_age, bath_sqft_interaction, and lot_living_ratio were kept in the best subsets. This suggests that they added useful information, especially for capturing nonlinear patterns, even though feature selection itself did not lead to a large performance boost.

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [7]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make sure Part 3 has already been run.
required_vars = ["X_train_fe", "X_test_fe", "y_train", "y_test", "best_feature_subsets"]
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise NameError(f"Run the Part 3 cell first. Missing variables: {missing_vars}")

cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=random_state)
part4_results = []
best_tuned_models = {}

for model_name in ["Linear Regression", "Ridge Regression", "Random Forest"]:
    selected_features = best_feature_subsets[model_name]
    X_train_model = X_train_fe[selected_features].copy()
    X_test_model = X_test_fe[selected_features].copy()

    # Scale linear models, but keep Random Forest on the raw selected features.
    if model_name in ["Linear Regression", "Ridge Regression"]:
        scaler_model = StandardScaler()
        X_train_model = scaler_model.fit_transform(X_train_model)
        X_test_model = scaler_model.transform(X_test_model)

    if model_name == "Linear Regression":
        estimator = LinearRegression()
        param_grid = {
            "fit_intercept": [True, False],
            "positive": [False, True],
        }
    elif model_name == "Ridge Regression":
        estimator = Ridge(random_state=random_state)
        param_grid = {
            "alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
            "fit_intercept": [True, False],
        }
    else:
        estimator = RandomForestRegressor(random_state=random_state, n_jobs=-1)
        param_grid = {
            "n_estimators": [100, 200],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2],
        }

    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )

    grid_search.fit(X_train_model, y_train)
    best_model = grid_search.best_estimator_
    best_tuned_models[model_name] = best_model

    best_cv_mae = -grid_search.best_score_
    best_cv_std = grid_search.cv_results_["std_test_score"][grid_search.best_index_]

    train_pred = best_model.predict(X_train_model)
    test_pred = best_model.predict(X_test_model)

    part4_results.append({
        "Model": model_name,
        "Selected Features": len(selected_features),
        "Best Parameters": str(grid_search.best_params_),
        "CV MAE Mean": best_cv_mae,
        "CV MAE Std": best_cv_std,
        "Train MAE": mean_absolute_error(y_train, train_pred),
        "Test MAE": mean_absolute_error(y_test, test_pred),
        "Train RMSE": mean_squared_error(y_train, train_pred) ** 0.5,
        "Test RMSE": mean_squared_error(y_test, test_pred) ** 0.5,
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred),
    })

part4_results = pd.DataFrame(part4_results).sort_values(by="CV MAE Mean")
display(part4_results.round({
    "CV MAE Mean": 2,
    "CV MAE Std": 2,
    "Train MAE": 2,
    "Test MAE": 2,
    "Train RMSE": 2,
    "Test RMSE": 2,
    "Train R2": 4,
    "Test R2": 4,
}))

best_model_part4 = part4_results.iloc[0]["Model"]
print(f"Best tuned model by CV MAE: {best_model_part4}")


/Users/bunphengchhay/miniforge3/envs/bu_env/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
3 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/bunphengchhay/miniforge3/envs/bu_env/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/bunphengchhay/miniforge3/envs/bu_env/lib/python3.12/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/bunphengchhay/mi

,Model,Selected Features,Best Parameters,CV MAE Mean,CV MAE Std,Train MAE,Test MAE,Train RMSE,Test RMSE,Train R2,Test R2
2,Random Forest,23,"{'max_depth': 20, 'min_samples_leaf': 2, 'min_...",189956.93,3647.88,111350.46,187480.34,210753.27,477652.74,0.8897,0.5649
0,Linear Regression,23,"{'fit_intercept': True, 'positive': False}",233795.46,3985.25,233416.79,228015.81,472370.49,490204.24,0.4458,0.5418
1,Ridge Regression,23,"{'alpha': 0.01, 'fit_intercept': True}",233795.47,3985.25,233416.79,228015.80,472370.49,490204.33,0.4458,0.5418


Best tuned model by CV MAE: Random Forest


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


**Tuning Strategy and Hyperparameter Selection:**

For **Linear Regression**, hyperparameter tuning was relatively limited since this model has only two main parameters: `fit_intercept` and `positive`. The tuning strategy involved testing whether to fit an intercept and whether to restrict coefficients to be non-negative. Both parameters were tested (fit_intercept: True/False, positive: False/True) to explore whether constraining the model would improve generalization. The best parameters were fit_intercept=True and positive=False, which allows the model to learn an intercept and permits both positive and negative coefficients. This is the more flexible approach and makes sense for housing price prediction, where features can have complex positive or negative relationships with the target.

For **Ridge Regression**, the primary tuning focus was the regularization strength (`alpha`), with values tested ranging from 0.01 to 100.0, combined with the `fit_intercept` parameter. Ridge regression adds an L2 penalty to shrink coefficients and reduce overfitting. The best alpha value found was 0.01—a very small penalty—suggesting that strong regularization was not needed for this dataset. This makes sense because the linear models already showed relatively stable, realistic predictions (low train-test gap) compared to Random Forest. The low alpha indicates the model could learn most of the useful relationships without needing much constraint.

For **Random Forest**, hyperparameter tuning was more extensive, exploring:
- `n_estimators` (100, 200): Number of trees in the forest
- `max_depth` (None, 10, 20): Maximum depth of each tree
- `min_samples_split` (2, 5): Minimum samples required to split a node
- `min_samples_leaf` (1, 2): Minimum samples required at leaf nodes

The best parameters found were n_estimators=200, max_depth=20, min_samples_split=2, and min_samples_leaf=2. The reasoning behind this tuning was to balance model complexity and generalization. By allowing deeper trees and more leaves, the model can capture complex patterns in the data. The relatively high n_estimators (200) builds a robust ensemble, while moderate max_depth (20) prevents complete overfitting while still allowing the model to learn non-linear relationships that linear models cannot capture.

**Feature Engineering and Model-Specific Performance:**

The results clearly show that feature engineering worked very differently for different model types. **Linear models (Linear Regression and Ridge Regression) benefited significantly from engineered features.** When comparing Part 1 (no engineering) to Part 2 (with engineered features), their CV MAE improved from ~246,555 to ~233,795—a meaningful improvement of about 5%. The engineered features (especially `sqft_squared` and `bath_sqft_interaction`) helped linear models because they cannot learn nonlinear transformations on their own. By manually providing squared square footage and interaction terms, we gave the linear models access to higher-order relationships.

In contrast, **Random Forest showed minimal improvement with feature engineering.** Its CV MAE actually slightly worsened from 189,957 (Part 1) to 191,823 (Part 2), though test R² improved marginally from 0.5569 to 0.5578. This is because tree-based models inherently learn nonlinear relationships and interactions by splitting on features at different thresholds. Adding pre-computed engineered features provides little additional benefit and may even introduce noise or redundancy. This demonstrates an important principle: the value of feature engineering depends on the model's inductive bias.

Interestingly, feature selection in Part 3 did not significantly change performance for any model, suggesting that the best features were already well-represented in the engineered feature set. The final tuned models maintained 23 selected features, indicating that careful feature engineering in Part 2 had already captured the most valuable information. Random Forest remained the clear winner by CV MAE throughout all stages, from baseline (189,957) through tuning (189,956.93), confirming that tree-based models are generally more effective for this housing price prediction task than linear models, even after tuning and feature engineering.

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [8]:
# PART 5: Final Model Selection and Results
# Reuse the best tuned model from Part 4 (already trained - no retraining needed)

# The best model by CV MAE
final_model_name = part4_results.iloc[0]["Model"]
final_model = best_tuned_models[final_model_name]
final_features = best_feature_subsets[final_model_name]

print("="*60)
print(f"FINAL MODEL: {final_model_name}")
print("="*60)

# Get data for final model
X_train_final = X_train_fe[final_features].copy()
X_test_final = X_test_fe[final_features].copy()

# Scale if linear model
if final_model_name in ["Linear Regression", "Ridge Regression"]:
    scaler_final = StandardScaler()
    X_train_final = scaler_final.fit_transform(X_train_final)
    X_test_final = scaler_final.transform(X_test_final)

# Extract metrics from Part 4 results
cv_results = part4_results[part4_results["Model"] == final_model_name].iloc[0]

print(f"\n CROSS-VALIDATION PERFORMANCE (5-fold, 3-repeat):")
print(f"   CV MAE Mean:     ${cv_results['CV MAE Mean']:,.2f}")
print(f"   CV MAE Std Dev:  ${cv_results['CV MAE Std']:,.2f}")

print(f"\n TRAINING SET PERFORMANCE:")
print(f"   Train MAE:   ${cv_results['Train MAE']:,.2f}")
print(f"   Train RMSE:  ${cv_results['Train RMSE']:,.2f}")
print(f"   Train R²:    {cv_results['Train R2']:.4f}")

print(f"\n TEST SET PERFORMANCE (Held-out):")
print(f"   Test MAE:    ${cv_results['Test MAE']:,.2f}")
print(f"   Test RMSE:   ${cv_results['Test RMSE']:,.2f}")
print(f"   Test R²:     {cv_results['Test R2']:.4f}")

print(f"\n BEST HYPERPARAMETERS:")
print(f"   {cv_results['Best Parameters']}")

print(f"\n MODEL SPECIFICATIONS:")
print(f"   Number of Features: {cv_results['Selected Features']}")
print(f"   Features Used: {final_features}")

FINAL MODEL: Random Forest

 CROSS-VALIDATION PERFORMANCE (5-fold, 3-repeat):
   CV MAE Mean:     $189,956.93
   CV MAE Std Dev:  $3,647.88

 TRAINING SET PERFORMANCE:
   Train MAE:   $111,350.46
   Train RMSE:  $210,753.27
   Train R²:    0.8897

 TEST SET PERFORMANCE (Held-out):
   Test MAE:    $187,480.34
   Test RMSE:   $477,652.74
   Test R²:     0.5649

 BEST HYPERPARAMETERS:
   {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}

 MODEL SPECIFICATIONS:
   Number of Features: 23
   Features Used: ['finishedsquarefeet12', 'regionidzip', 'latitude', 'longitude', 'bath_sqft_interaction', 'sqft_squared', 'calculatedfinishedsquarefeet', 'lotsizesquarefeet', 'lot_living_ratio', 'buildingqualitytypeid', 'yearbuilt', 'home_age', 'propertyzoningdesc', 'regionidcity', 'bedroomcnt', 'propertycountylandusecode', 'heatingorsystemtypeid', 'bathroomcnt', 'calculatedbathnbr', 'fullbathcnt', 'roomcnt', 'propertylandusetypeid', 'unitcnt']


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

**1. Model Selection**

I selected **Random Forest** as the final model. The decision was driven primarily by **lowest CV MAE ($189,956.93)** across all tuning stages (Part 1 baseline through Part 4), consistently outperforming Linear and Ridge Regression by ~$44k. Test R² of 0.5649 was also the highest, explaining over 56% of price variance on unseen data. The trade-off was acknowledgment of overfitting (train R² = 0.8897 vs test R² = 0.5649), but this gap is acceptable for an ensemble model and beats the alternatives. Interpretability was sacrificed for predictive power, a necessary choice given the dataset's complexity.

**2. Revisiting Early Preprocessing Decision: Missing Value Imputation**

In Milestone 1, I chose **median imputation** for numerical missing values (rather than deletion or forward-fill) to preserve sample size and avoid bias from non-random missingness patterns common in real estate data. I hoped this would retain predictive signal while maintaining data integrity.

The decision proved **sound**: features like yearbuilt, calculatedfinishedsquarefeet, and bathroomcnt—all imputed with median—ended up in the final feature set, contributing meaningfully to the model. Median imputation was robust because housing prices have right skewed distributions, making median less sensitive to outliers than mean imputation would have been. The fact that Random Forest selected 23 features (many originally imputed) and achieved strong test performance validates this choice. I kept it unchanged throughout the pipeline.

**3. Lessons Learned**

**Key insights:**
- **Tree models dominate linear models on this dataset**  the inherent nonlinearity of housing prices (location effects, nonlinear square footage impact) favors ensembles over simple regression.
- **Feature engineering helps linear models more than tree models** manually engineering sqft_squared and bath_sqft_interaction boosted Linear Regression by ~5%, but Random Forest barely improved. This highlights the importance of matching preprocessing strategy to model architecture.
- **Feature selection provided marginal gains** reducing from all features to 23 didn't significantly improve performance, suggesting the engineered features were already efficient.

**Future directions:** With more time, I would explore (1) **ensemble stacking** (combining RF with linear models) to capture both nonlinear and linear relationships; (2) **hyperparameter tuning with Bayesian optimization** for finer grained search; (3) **domain-specific features** like neighborhood walkability or school district proximity if external data were available; and (4) **interaction terms for tree models** as explicit features to see if Random Forest can leverage hand crafted interactions better than currently.